In [2]:
import pandas as pd
import numpy as np
from scipy.special import expit, logsumexp
import time

def get_log_lik_vec(beta, X, y):
    logits = X @ beta.T  
    ll = np.sum(y[:, None] * logits - np.log1p(np.exp(logits)), axis=0)
    return ll

def get_grad_log_lik_vec(beta, X, y):
    logits = X @ beta.T         
    probs = expit(logits)        
    diff = y[:, None] - probs    
    grad = diff.T @ X            
    return grad

def get_log_prior_vec(beta, prior_var=10.0):
    d = beta.shape[1]
    const = - (d / 2.0) * np.log(2 * np.pi * prior_var)
    return const - (1.0 / (2 * prior_var)) * np.sum(beta**2, axis=1) 

def get_grad_log_prior_vec(beta, prior_var=10.0):
    return - beta / prior_var

def hmc_transition_step_vec(beta_current, t, epsilon, L, X, y, prior_var=10.0):
    M, d = beta_current.shape
    p_current = np.random.normal(0, 1, size=(M, d))
    
    beta = beta_current.copy()
    p = p_current.copy()
    
    def get_grad_U(b):
        return -(get_grad_log_prior_vec(b, prior_var) + t * get_grad_log_lik_vec(b, X, y))
    
    grad_U = get_grad_U(beta)
    p = p - 0.5 * epsilon * grad_U
    
    for i in range(L):
        beta = beta + epsilon * p
        grad_U = get_grad_U(beta)
        if i != L - 1:
            p = p - epsilon * grad_U
            
    p = p - 0.5 * epsilon * grad_U
    
    U_current = -(get_log_prior_vec(beta_current, prior_var) + t * get_log_lik_vec(beta_current, X, y))
    K_current = 0.5 * np.sum(p_current**2, axis=1)
    
    U_proposed = -(get_log_prior_vec(beta, prior_var) + t * get_log_lik_vec(beta, X, y))
    K_proposed = 0.5 * np.sum(p**2, axis=1)
    
    energy_diff = (U_current + K_current) - (U_proposed + K_proposed)
    alpha = np.exp(np.clip(energy_diff, -100, 0))
    
    accept = np.random.rand(M) < alpha
    beta_new = np.where(accept[:, None], beta, beta_current)
    
    return beta_new, np.sum(accept)


def run_ais_hmc_vec(X, y, num_chains, num_steps, epsilon, L, prior_var=10.0, verbose=True):
    d = X.shape[1]
    temperatures = np.linspace(0, 1, num_steps)
    
    log_weights = np.zeros(num_chains)
    beta = np.random.normal(0, np.sqrt(prior_var), size=(num_chains, d))
    
    phase_accepts = np.zeros(3)
    phase_transitions = np.zeros(3)
    ti_mean_log_likelihood = np.zeros(num_steps)
    
    total_transitions = num_chains * (num_steps - 1)
    total_grad_evals = total_transitions * (L + 1)
    
    start_time = time.time()
    
    for k in range(1, num_steps):
        t_prev = temperatures[k-1]
        t_curr = temperatures[k]
        
        current_log_lik = get_log_lik_vec(beta, X, y)
        log_weights += (t_curr - t_prev) * current_log_lik
        ti_mean_log_likelihood[k] = np.mean(current_log_lik)
        
        beta, accepted_count = hmc_transition_step_vec(beta, t_curr, epsilon, L, X, y, prior_var)
        
        if t_curr < 0.33:
            phase = 0
        elif t_curr < 0.67:
            phase = 1
        else:
            phase = 2
            
        phase_accepts[phase] += accepted_count
        phase_transitions[phase] += num_chains
        
        if verbose and k % max(1, num_steps // 50) == 0:
            print(f" 退火演进: {k}/{num_steps} 层")

    runtime = time.time() - start_time
    
    log_marginal_likelihood = logsumexp(log_weights) - np.log(num_chains)
    log_ess = 2 * logsumexp(log_weights) - logsumexp(2 * log_weights)
    ess = np.exp(log_ess)
    var_log_w = np.var(log_weights)
    
    ti_mean_log_likelihood = ti_mean_log_likelihood
    delta_t = 1.0 / (num_steps - 1)
    ti_estimate = np.sum(ti_mean_log_likelihood[1:] * delta_t)
    phase_rates = (phase_accepts / phase_transitions) * 100
    
    return log_marginal_likelihood, runtime, ess, var_log_w, phase_rates, ti_estimate, total_transitions, total_grad_evals


def run_macro_trials(dataset_name, X, y, num_trials, num_chains, num_steps, epsilon, L):
    
    log_z_list = []
    runtime_list = []
    ess_list = []
    
    actual_trans_count = num_chains * (num_steps - 1)
    actual_grad_count = actual_trans_count * (L + 1)
    
    for i in range(1, num_trials + 1):
        print(f"Trial {i}/{num_trials}", end="", flush=True)
        
        log_z, runtime, ess, var_w, phase, ti_est, _, _ = run_ais_hmc_vec(
            X=X, y=y, num_chains=num_chains, num_steps=num_steps, 
            epsilon=epsilon, L=L, verbose=False
        )
        
        log_z_list.append(log_z)
        runtime_list.append(runtime)
        ess_list.append(ess)
        print(f" log p(y) = {log_z:.4f} | 物理耗时 = {runtime:.2f}s")
        
    mean_log_z = np.mean(log_z_list)
    std_log_z = np.std(log_z_list, ddof=1)  
    mean_runtime = np.mean(runtime_list)
    mean_ess = np.mean(ess_list)
    
    print("汇总:")
    print(f"边际似然估计: 均值 = {mean_log_z:.4f} | 经验标准差(稳定性) = ±{std_log_z:.4f}")
    print(f"⏱平均耗时: {mean_runtime:.2f} 秒 / 次")
    print(f"平均有效样本: {mean_ess:.2f} / {num_chains} (ESS)")
    print(f"单次硬件无关算力成本: {actual_trans_count} 次 HMC 转移 | {actual_grad_count} 次底层梯度评估")
    
    return mean_log_z, std_log_z, mean_runtime, actual_grad_count

def load_and_split_data(csv_file_path):
    df = pd.read_csv(csv_file_path)
    y = df['y'].values
    X = df.drop(columns=['y']).values
    return X, y

if __name__ == "__main__":
    NUM_TRIALS = 30

    print("低维后验场景")
    X_pima, y_pima = load_and_split_data("pima_preprocessed_低维.csv")
    run_macro_trials("低维场景", X_pima, y_pima, num_trials=NUM_TRIALS, num_chains=100, num_steps=500, epsilon=0.10, L=10)

低维后验场景
Trial 1/30 log p(y) = -388.4966 | 物理耗时 = 30.45s
Trial 2/30 log p(y) = -388.5898 | 物理耗时 = 29.99s
Trial 3/30 log p(y) = -388.2054 | 物理耗时 = 31.58s
Trial 4/30 log p(y) = -388.9203 | 物理耗时 = 30.30s
Trial 5/30 log p(y) = -389.6321 | 物理耗时 = 28.92s
Trial 6/30 log p(y) = -388.9645 | 物理耗时 = 29.12s
Trial 7/30 log p(y) = -389.6918 | 物理耗时 = 29.15s
Trial 8/30 log p(y) = -387.6469 | 物理耗时 = 30.35s
Trial 9/30 log p(y) = -388.0695 | 物理耗时 = 59.11s
Trial 10/30 log p(y) = -385.5599 | 物理耗时 = 37.72s
Trial 11/30 log p(y) = -388.9541 | 物理耗时 = 45.68s
Trial 12/30 log p(y) = -388.4481 | 物理耗时 = 45.22s
Trial 13/30 log p(y) = -388.8606 | 物理耗时 = 63.28s
Trial 14/30 log p(y) = -389.1338 | 物理耗时 = 65.61s
Trial 15/30 log p(y) = -388.5338 | 物理耗时 = 57.44s
Trial 16/30 log p(y) = -388.5660 | 物理耗时 = 62.67s
Trial 17/30 log p(y) = -391.2087 | 物理耗时 = 51.89s
Trial 18/30 log p(y) = -388.3655 | 物理耗时 = 50.71s
Trial 19/30 log p(y) = -389.5832 | 物理耗时 = 49.52s
Trial 20/30 log p(y) = -388.7450 | 物理耗时 = 48.64s
Trial 21/30 log p(y) =

In [4]:
if __name__ == "__main__":
    NUM_TRIALS = 30
    print("中维后验场景")
    X_credit, y_credit = load_and_split_data("creditcard_preprocessed_中维.csv")
    run_macro_trials("中维场景", X_credit, y_credit, num_trials=NUM_TRIALS, num_chains=100, num_steps=500, epsilon=0.15, L=10)

中维后验场景
Trial 1/30

 log p(y) = -96.4609 | 物理耗时 = 30.10s
Trial 2/30 log p(y) = -100.8098 | 物理耗时 = 28.87s
Trial 3/30 log p(y) = -100.8589 | 物理耗时 = 30.04s
Trial 4/30 log p(y) = -100.5374 | 物理耗时 = 30.27s
Trial 5/30 log p(y) = -100.7560 | 物理耗时 = 29.66s
Trial 6/30 log p(y) = -101.0059 | 物理耗时 = 29.46s
Trial 7/30 log p(y) = -99.9815 | 物理耗时 = 40.00s
Trial 8/30 log p(y) = -100.2220 | 物理耗时 = 32.71s
Trial 9/30 log p(y) = -101.3727 | 物理耗时 = 30.07s
Trial 10/30 log p(y) = -100.6124 | 物理耗时 = 31.44s
Trial 11/30 log p(y) = -99.2039 | 物理耗时 = 28.70s
Trial 12/30 log p(y) = -96.5513 | 物理耗时 = 30.09s
Trial 13/30 log p(y) = -100.5706 | 物理耗时 = 28.91s
Trial 14/30 log p(y) = -100.6859 | 物理耗时 = 31.00s
Trial 15/30 log p(y) = -99.1862 | 物理耗时 = 28.44s
Trial 16/30 log p(y) = -97.3494 | 物理耗时 = 31.00s
Trial 17/30 log p(y) = -98.4394 | 物理耗时 = 29.51s
Trial 18/30 log p(y) = -101.5642 | 物理耗时 = 30.99s
Trial 19/30 log p(y) = -101.1118 | 物理耗时 = 28.26s
Trial 20/30 log p(y) = -100.6448 | 物理耗时 = 29.21s
Trial 21/30 log p(y) = -99.8496 | 物理耗时 = 29.92

In [5]:
if __name__ == "__main__":
    NUM_TRIALS = 30
    print("高维后验场景")
    X_tcga, y_tcga = load_and_split_data("tcga_preprocessed_高维.csv")
    run_macro_trials("高维场景", X_tcga, y_tcga, num_trials=NUM_TRIALS, num_chains=200, num_steps=1000, epsilon=0.06, L=15)

高维后验场景
Trial 1/30 log p(y) = -91.2410 | 物理耗时 = 558.84s
Trial 2/30 log p(y) = -88.3267 | 物理耗时 = 428.22s
Trial 3/30 log p(y) = -90.9459 | 物理耗时 = 736.66s
Trial 4/30 log p(y) = -84.1260 | 物理耗时 = 733.83s
Trial 5/30 log p(y) = -85.2319 | 物理耗时 = 612.16s
Trial 6/30 log p(y) = -78.0576 | 物理耗时 = 236.07s
Trial 7/30 log p(y) = -91.6841 | 物理耗时 = 630.26s
Trial 8/30 log p(y) = -83.8221 | 物理耗时 = 680.25s
Trial 9/30 log p(y) = -83.7110 | 物理耗时 = 613.47s
Trial 10/30 log p(y) = -89.3681 | 物理耗时 = 659.30s
Trial 11/30 log p(y) = -82.3357 | 物理耗时 = 1156.54s
Trial 12/30 log p(y) = -77.6398 | 物理耗时 = 655.37s
Trial 13/30 log p(y) = -87.6253 | 物理耗时 = 659.77s
Trial 14/30 log p(y) = -92.8435 | 物理耗时 = 2974.52s
Trial 15/30 log p(y) = -92.8340 | 物理耗时 = 633.19s
Trial 16/30 log p(y) = -81.4385 | 物理耗时 = 372.72s
Trial 17/30 log p(y) = -91.9633 | 物理耗时 = 270.48s
Trial 18/30 log p(y) = -87.9265 | 物理耗时 = 263.24s
Trial 19/30 log p(y) = -89.4448 | 物理耗时 = 228.07s
Trial 20/30 log p(y) = -88.5391 | 物理耗时 = 416.81s
Trial 21/30 log p(y)